# Grocery Spending Analysis

This notebook analyzes trips logged in my [Grocery Run Estimator](https://visualkumar.github.io/grocery-tracker/) app.

**Questions**
1. How much do I spend per month, and how does it split between Costco and Walmart?
2. Which categories take up most of my budget?
3. How accurate are my pre-trip estimates?
4. How do prices for individual items change over time?
5. Which items drive most of my spending?
6. What does protein cost me per gram, by source? (stretch)

**Data:** two CSV exports from the app. `trips` has one row per store per trip. `lines` has one row per item bought.
Until I have enough real trips, the analysis runs on generated sample data (`is_sample == True`).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Switch to False once real exports are in data/real/
USE_SAMPLE = True

DATA_DIR = Path("../data") / ("sample" if USE_SAMPLE else "real")
pd.set_option("display.float_format", "{:,.2f}".format)
DATA_DIR

## 1. Load and inspect

In [ ]:
trips_file = sorted(DATA_DIR.glob("*trips*.csv"))[-1]
lines_file = sorted(DATA_DIR.glob("*line_items*.csv"))[-1]

trips = pd.read_csv(trips_file, parse_dates=["date"])
lines = pd.read_csv(lines_file, parse_dates=["date"])

print(trips_file.name, trips.shape)
print(lines_file.name, lines.shape)
trips.head()

In [ ]:
trips.info()
print()
lines.info()

## 2. Data quality checks

Before analyzing, confirm the two files agree with each other. If any check fails, fix the data before trusting the results.

In [ ]:
# Each trip ID appears once in trips
assert trips["trip_id"].is_unique, "Duplicate trip IDs"

# Every line item belongs to a known trip
missing = set(lines["trip_id"]) - set(trips["trip_id"])
assert not missing, f"Line items with no matching trip: {missing}"

# Line totals add up to each trip's estimated total (allow a cent of rounding)
check = lines.groupby("trip_id")["line_total"].sum().rename("lines_sum")
check = trips.set_index("trip_id")[["est_total"]].join(check)
bad = check[(check["est_total"] - check["lines_sum"]).abs() > 0.02]
assert bad.empty, f"Totals don't match:\n{bad}"

# Item counts agree
counts = lines.groupby("trip_id")["qty"].sum().rename("lines_qty")
counts = trips.set_index("trip_id")[["item_count"]].join(counts)
assert (counts["item_count"] == counts["lines_qty"]).all(), "Item counts don't match"

print("All checks passed")
print("Trips missing an actual total:", trips["actual_total"].isna().sum())

### Prepare a spending column

`actual_total` is optional in the app. For spending questions, use what I actually paid when it's recorded and fall back to the estimate when it isn't. Keep this rule in mind when you describe results: fallback rows exclude tax.

In [ ]:
trips["spent"] = trips["actual_total"].fillna(trips["est_total"])
trips["month"] = trips["date"].dt.to_period("M")
lines["month"] = lines["date"].dt.to_period("M")
trips[["trip_id", "date", "store", "est_total", "actual_total", "spent"]].tail()

## Q1. Monthly spending by store

**Goal:** a table with one row per month and one column per store, plus a total. Then a stacked bar chart.

*Hints:* `pivot_table` with `index="month"`, `columns="store"`, `values="spent"`, `aggfunc="sum"`. A partial current month will look low, so note that when you describe the chart.

In [ ]:
# Your code here


**Takeaway:** _Write 1–2 sentences on what the chart shows._

## Q2. Spending by category

**Goal:** total `line_total` per category, sorted, with each category's share of the total. Then a horizontal bar chart.

*Hints:* `groupby("category")`, then divide by the grand total for the share. Line totals are pre-tax estimates, so shares are more reliable than dollar amounts here.

In [ ]:
# Your code here


**Takeaway:** _Which categories dominate? Is anything surprising?_

## Q3. How accurate are my estimates?

**Goal:** for trips with an `actual_total`, calculate `actual_total - est_total` and the percentage gap. Summarize by store (mean, median, count).

*Things to think about:* the gap includes sales tax, which differs by item type in Pennsylvania (most groceries aren't taxed, household items are). Would you expect Costco or Walmart trips to show a bigger gap, and why?

In [ ]:
# Your code here


**Takeaway:** _Is the gap consistent enough to add a correction factor to the app's estimate?_

## Q4. Price changes over time

**Goal:** pick one item bought on several trips and plot its `unit_price` by date. Then find the items whose price varies most.

*Hints:* filter with `lines[lines["item_name"] == ...]`. For variation, group by `item_name` and compute the coefficient of variation (`std / mean`), keeping only items bought at least 3 times. Exclude rows where `price_is_estimate` is True if you only want prices you've confirmed.

In [ ]:
# Your code here


**Takeaway:** _Which prices move the most? Is it real price change or estimation noise?_

## Q5. Which items drive my spending?

**Goal:** rank items by total spend and by how often they're bought. What share of spending comes from the top 5 items?

*Hints:* `groupby("item_name").agg(...)` with `sum` and `count`, then `cumsum()` on the sorted share for a Pareto view.

In [ ]:
# Your code here


**Takeaway:** _Where would a price change or a store switch save the most money?_

## Q6 (stretch). Cost per gram of protein

**Goal:** compare protein sources by dollars per gram of protein.

The export doesn't include nutrition data, so build a small table yourself from package labels: item name and total grams of protein per package (or per pound for meat). Merge it onto `lines` and compute cost per gram.

*Hints:* `pd.DataFrame({...})` for the lookup table, `merge(..., on="item_name")`. For per-pound items, total protein depends on `lbs`.

In [ ]:
# Your code here


## Findings

_Summarize 3–5 findings here once the analysis is done, with the numbers that support each one. Note any limits of the data, such as sample data, missing actual totals, or estimated prices._